# Notebook 2 — Modern Semantic RAG
## Fixing the failures you measured in Notebook 1

**Where we left off.** Notebook 1 ended with a flat line: `hit@1 ≈ 0.5` at *every* chunk size and
overlap. Five lexical questions always hit, five paraphrased questions always missed. The diagnosis:

> TF-IDF matches **strings**, not **meaning**. `"The message bus is falling behind"` scored **0.000**
> against the Kafka runbook because it shares zero tokens with it.

This notebook fixes that — and then discovers the *next* failure underneath it.

**What we add, and which specific failure each part repairs:**

| Component | Fixes |
|---|---|
| **Sentence-transformer embeddings** | synonym blindness ("message bus" ≈ "Kafka") |
| **FAISS index** | brute-force for-loop doesn't scale past ~10k vectors |
| **Cross-encoder re-ranking** | vocabulary collisions ("failures at checkout" → wrong runbook) |
| **Hybrid BM25 + vector** | *semantic* search loses exact identifiers (`payment-api-0`, error codes) |
| **Metadata filtering** | during a Kafka incident, don't search login runbooks at all |

**Rules unchanged:** still no LangChain. Every component stays inspectable.

> ⚠️ **Runtime:** use a CPU runtime — that's plenty. The first cell downloads ~120 MB of models
> (MiniLM + a cross-encoder). Takes ~1 minute on Colab.


---
## Section 0 — Setup

`sentence-transformers` brings in torch; `faiss-cpu` is the vector index; `rank_bm25` is the keyword scorer.


In [ ]:
%pip install -q sentence-transformers faiss-cpu rank_bm25

In [ ]:
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_colwidth", 70)
plt.rcParams["figure.figsize"] = (10, 5)

print("Setup complete ✅")

---
## Section 1 — Rebuild the Notebook 1 pipeline (identical corpus)

For an honest before/after comparison, the corpus, cleaning, and chunking must be **byte-identical**
to Notebook 1. Only the *embedding* changes. This is basic experimental hygiene: change one variable
at a time, or you can't attribute the improvement.


In [ ]:
RAW_RUNBOOKS = {
    "payment-service": """
CONFIDENTIAL - ACME Corp Internal
Page 1 of 2

RUNBOOK: Payment Service Outage

Symptoms:
- 5xx errors on /api/v1/charge endpoint
- Latency above 2000ms on payment-api
- Redis connection timeouts in payment logs

Investigation steps:
1. Check Kubernetes pods for the payment namespace.
   kubectl get pods -n payment
2. Inspect recent logs for exceptions.
   kubectl logs deployment/payment-api -n payment --tail=200
3. Verify Redis connectivity from a payment pod.
   kubectl exec -it payment-api-0 -- redis-cli -h redis-payment ping

Remediation:
If pods are in CrashLoopBackOff, restart the deployment:
   kubectl rollout restart deployment payment-api -n payment
If Redis is unreachable, failover to the replica:
   redis-cli -h redis-payment SENTINEL failover payment-master

Escalation: page the payments on-call if errors persist 15 minutes.

CONFIDENTIAL - ACME Corp Internal
Page 2 of 2
""",

    "order-service": """
CONFIDENTIAL - ACME Corp Internal
Page 1 of 1

RUNBOOK: Order Service Degradation

Symptoms:
- Orders stuck in PENDING state
- Growing queue depth on order-events topic
- Timeouts calling payment-service downstream

Investigation steps:
1. Check order-service pod health.
   kubectl get pods -n orders
2. Check the dead letter queue for poison messages.
   kubectl logs deployment/order-worker -n orders | grep DLQ
3. Verify downstream payment-service is healthy before restarting anything.

Remediation:
Restart the order worker to reprocess stuck orders:
   kubectl rollout restart deployment order-worker -n orders
Replay dead-lettered events after fixing the schema issue.

CONFIDENTIAL - ACME Corp Internal
""",

    "login-service": """
CONFIDENTIAL - ACME Corp Internal

RUNBOOK: Login Service Authentication Failures

Symptoms:
- Spike in 401 responses on /auth/login
- JWT signature validation errors in logs
- Session store (Redis) memory above 90 percent

Investigation steps:
1. Confirm the JWT signing key was not rotated without deployment.
   kubectl get secret jwt-signing-key -n auth -o yaml
2. Check Redis session store memory.
   redis-cli -h redis-sessions INFO memory
3. Review recent deployments to login-service.

Remediation:
Roll back the last deployment if key rotation caused the failure:
   kubectl rollout undo deployment login-service -n auth
Evict expired sessions if Redis memory is exhausted.

Page the identity team if MFA providers are timing out.

CONFIDENTIAL - ACME Corp Internal
""",

    "kafka-cluster": """
CONFIDENTIAL - ACME Corp Internal
Page 1 of 2

RUNBOOK: Kafka Cluster Incidents

Symptoms:
- Consumer lag increasing on critical topics
- Under-replicated partitions alert firing
- ISR shrinking on broker 2
- Producers receiving NotEnoughReplicasException

Investigation steps:
1. Check broker health and disk usage.
   kafka-broker-api-versions.sh --bootstrap-server kafka-0:9092
2. List under-replicated partitions.
   kafka-topics.sh --describe --under-replicated-partitions --bootstrap-server kafka-0:9092
3. Check consumer group lag.
   kafka-consumer-groups.sh --describe --group order-consumers --bootstrap-server kafka-0:9092

Remediation:
If a broker is down due to disk pressure, clear old log segments and restart the broker:
   systemctl restart kafka
If consumer lag keeps growing, scale the consumer group before touching the brokers.
Never delete topics during an incident.

CONFIDENTIAL - ACME Corp Internal
Page 2 of 2
""",

    "kubernetes-deploys": """
CONFIDENTIAL - ACME Corp Internal

RUNBOOK: Kubernetes Deployment Failures

Symptoms:
- Pods stuck in ImagePullBackOff or CrashLoopBackOff
- Rollout stuck at 50 percent
- Readiness probes failing after a new release

Investigation steps:
1. Describe the failing pod to see events.
   kubectl describe pod <pod-name>
2. Check rollout status.
   kubectl rollout status deployment <name>
3. Compare the new image tag against the registry.

Remediation:
Roll back a bad release immediately:
   kubectl rollout undo deployment <name>
Fix readiness probe thresholds if the app needs longer warmup.
Always roll back first, debug second, during a customer-facing incident.

CONFIDENTIAL - ACME Corp Internal
""",
}

HEADER_FOOTER_PATTERNS = [r"CONFIDENTIAL.*", r"Page \d+ of \d+"]

def clean(text: str) -> str:
    for pat in HEADER_FOOTER_PATTERNS:
        text = re.sub(pat, "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def chunk_words(text: str, chunk_size: int, overlap: int = 0):
    assert overlap < chunk_size
    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        piece = words[start : start + chunk_size]
        if len(piece) < 5:
            break
        chunks.append(" ".join(piece))
        if start + chunk_size >= len(words):
            break
    return chunks

CLEAN_RUNBOOKS = {n: clean(t) for n, t in RAW_RUNBOOKS.items()}
CHUNK_SIZE, OVERLAP = 60, 12

corpus = []
for doc_name, text in CLEAN_RUNBOOKS.items():
    for i, ch in enumerate(chunk_words(text, CHUNK_SIZE, OVERLAP)):
        corpus.append({"doc": doc_name, "chunk_id": f"{doc_name}#{i}", "text": ch})

df = pd.DataFrame(corpus)
print(f"{len(df)} chunks — identical to Notebook 1")
df.head()

### 1.1 The evaluation set (also identical)

Deliberately split into a **lexical** half and a **semantic** half. This split is the instrument that
tells us *which* component is the bottleneck — the single most valuable idea in these two notebooks.


In [ ]:
TEST_SET = [
    # -------- lexical: share vocabulary with the runbook --------
    ("How do I restart the payment service?",       "payment-service",    "lexical"),
    ("Why is Kafka consumer lag increasing?",       "kafka-cluster",      "lexical"),
    ("Users are getting 401 errors on login",       "login-service",      "lexical"),
    ("Orders are stuck in pending state",           "order-service",      "lexical"),
    ("How do I fix under-replicated partitions?",   "kafka-cluster",      "lexical"),
    # -------- semantic: paraphrases, zero/low token overlap --------
    ("The message bus is falling behind",           "kafka-cluster",      "semantic"),
    ("Sign-in is broken for everyone",              "login-service",      "semantic"),
    ("Customers see failures at checkout",          "payment-service",    "semantic"),
    ("The event pipeline stopped delivering data",  "kafka-cluster",      "semantic"),
    ("Purchases are timing out",                    "order-service",      "semantic"),
]
df_test = pd.DataFrame(TEST_SET, columns=["question", "expected_doc", "kind"])
df_test

---
## Section 2 — The Baseline (TF-IDF), reproduced

Let's re-establish Notebook 1's numbers so the improvement is measured, not asserted.


In [ ]:
tfidf = TfidfVectorizer(lowercase=True, stop_words="english",
                        token_pattern=r"[a-zA-Z][a-zA-Z\-]+")
X_tfidf = tfidf.fit_transform(df["text"]).toarray()   # same TF-IDF setup as Notebook 1, rebuilt here
                                                        # so this notebook is fully self-contained

def cosine_matrix(mat, q):
    # Vectorized cosine similarity: q is ONE query vector, mat is ALL chunk vectors stacked as rows.
    # mat @ q computes the dot product against every row in a single call (no Python loop needed);
    # dividing by each row's length times the query's length turns those dot products into cosines.
    denom = np.linalg.norm(mat, axis=1) * np.linalg.norm(q) + 1e-9   # +1e-9 avoids divide-by-zero
    return (mat @ q) / denom

def evaluate(score_fn, label):
    """Generic evaluation harness — works for ANY retriever, because it only needs a function
    that takes a question string and returns a similarity score for every chunk. This same
    function gets reused later for TF-IDF, dense, BM25, hybrid, and reranked retrievers —
    a good example of writing one measurement tool instead of duplicating it per retriever."""
    rows = []
    for q, expected, kind in TEST_SET:       # kind is "lexical" or "semantic" — see Section 1.1
        scores = score_fn(q)                 # score_fn does whatever that retriever's math is;
                                              # evaluate() itself doesn't need to know how
        best = int(np.argmax(scores))        # index of the single highest-scoring chunk
        rows.append({"question": q, "kind": kind, "expected": expected,
                     "got": df.loc[best, "doc"], "score": float(scores[best]),
                     "hit": df.loc[best, "doc"] == expected})   # True if top chunk's doc matches
    out = pd.DataFrame(rows)
    overall = out.hit.mean()                          # mean of a boolean column = fraction True
    by_kind = out.groupby("kind").hit.mean()           # same, but split by lexical vs semantic —
                                                         # this is what lets us see WHERE a retriever
                                                         # wins or loses instead of one blended number
    print(f"--- {label} ---")
    print(f"  hit@1 overall : {overall:.2f}")
    for k, v in by_kind.items():
        print(f"  hit@1 {k:9s}: {v:.2f}")
    return out

# lambda wraps cosine_matrix so it matches the score_fn(question) -> scores signature evaluate() expects
tfidf_scores = lambda q: cosine_matrix(X_tfidf, tfidf.transform([q]).toarray().ravel())
res_tfidf = evaluate(tfidf_scores, "TF-IDF baseline (Notebook 1)")
res_tfidf[["kind", "question", "expected", "got", "score", "hit"]]

---
## Section 3 — Sentence-Transformer Embeddings

**The conceptual leap.** TF-IDF assigns one dimension per *word*. A sentence-transformer maps the
whole sentence into a fixed ~384-dim space that was *trained* so that paraphrases land near each other.
Nothing about the pipeline changes — same chunks, same cosine similarity, same top-k. Only the
`text → vector` function is swapped.

`all-MiniLM-L6-v2`: 6 transformer layers, 384 dims, ~80 MB, fast on CPU. The workhorse default.

**Production note — `normalize_embeddings=True`.** It scales every vector to unit length, which makes
the **dot product mathematically identical to cosine similarity**. That matters in Section 4: FAISS's
fast inner-product index then gives us cosine for free. Skipping normalization here is a classic bug —
search still "works" but ranks by a mix of similarity *and* vector magnitude.


In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Inside .encode(), for EACH chunk: the text is tokenized (split into sub-word pieces),
# every token gets its own contextual vector from the transformer layers, and then all of
# those per-token vectors are MEAN-POOLED (averaged, column by column) into ONE 384-number
# vector — regardless of whether the chunk has 10 tokens or 250. That's why chunk length
# doesn't change the shape of X_dense below: pooling always collapses to a fixed size.
X_dense = embedder.encode(
    df["text"].tolist(),           # list of chunk texts in, list of vectors out — one per chunk
    normalize_embeddings=True,     # scales each vector to length 1.0, which makes dot product
                                    # mathematically equal to cosine similarity (see Section 4)
    show_progress_bar=True,
).astype("float32")                # float32 halves memory vs. numpy's default float64 — matters
                                    # once this scales past a handful of chunks

print(f"\nTF-IDF matrix : {X_tfidf.shape}   ({X_tfidf.shape[1]} readable word-dims, ~84% zeros)")
print(f"MiniLM matrix : {X_dense.shape}   (384 dense learned dims, ~0% zeros)")
# sanity check that normalize_embeddings actually did its job — every row's length should be ~1.0
print(f"\nUnit-norm check (should all be 1.0): {np.linalg.norm(X_dense, axis=1)[:5].round(4)}")

### 3.1 The moment of truth — does it fix synonym blindness?

Notebook 1 scored **0.000** on "The message bus is falling behind". Watch this specific query.


In [ ]:
def dense_scores(q):
    qv = embedder.encode([q], normalize_embeddings=True).astype("float32").ravel()
    # X_dense has shape (n_chunks, 384); qv has shape (384,). "@" (matrix-vector product) computes
    # the dot product of qv against EVERY ROW of X_dense in one call, returning shape (n_chunks,) —
    # this is the same idea as the mat @ qv line in Notebook 1's eval harness, just simpler here
    # because BOTH sides are already unit-length (normalize_embeddings=True), so dot == cosine
    # directly — no need to divide by norms the way Notebook 1's cosine() function did.
    return X_dense @ qv

probe = "The message bus is falling behind"
tf_s, dn_s = tfidf_scores(probe), dense_scores(probe)

comparison = pd.DataFrame({
    "chunk_id": df.chunk_id, "doc": df.doc,
    "tfidf": tf_s.round(3), "minilm": dn_s.round(3),
}).sort_values("minilm", ascending=False)

print(f'QUERY: "{probe}"   (expected: kafka-cluster)\n')
print(f"TF-IDF  best → {df.loc[int(tf_s.argmax()), 'doc']}  (score {tf_s.max():.3f})")
print(f"MiniLM  best → {df.loc[int(dn_s.argmax()), 'doc']}  (score {dn_s.max():.3f})")
comparison.head(6)

In [ ]:
# Side-by-side scores across all chunks for the same query
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
colors = ["#C44E52" if d == "kafka-cluster" else "#BBBBBB" for d in df.doc]
for ax, scores, name in zip(axes, [tf_s, dn_s], ["TF-IDF", "MiniLM"]):
    ax.barh(df.chunk_id, scores, color=colors)
    ax.set_title(f'{name}: "{probe}"')
    ax.set_xlabel("similarity"); ax.invert_yaxis()
axes[0].set_ylabel("chunk")
plt.suptitle("Red = the correct runbook (kafka-cluster)")
plt.tight_layout(); plt.show()

In [ ]:
res_dense = evaluate(dense_scores, "MiniLM dense embeddings")
res_dense[["kind", "question", "expected", "got", "score", "hit"]]

**Read the `by_kind` breakdown, not just the overall number.** The lexical half should stay high;
the semantic half is where the movement is. This is the payoff of splitting the eval set — you can
attribute the gain to the component you changed.


### 3.2 Re-draw the embedding map

Same PCA projection as Notebook 1 §5, now over learned vectors. Compare the two shapes: TF-IDF
clusters by *shared vocabulary*, MiniLM clusters by *topic*.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
palette = {"payment-service": "#4C72B0", "order-service": "#DD8452",
           "login-service": "#55A868", "kafka-cluster": "#C44E52",
           "kubernetes-deploys": "#8172B3"}

for ax, mat, name in zip(axes, [X_tfidf, X_dense], ["TF-IDF", "MiniLM"]):
    pts = PCA(n_components=2, random_state=42).fit_transform(mat)
    for doc_name, c in palette.items():
        m = (df.doc == doc_name).values
        ax.scatter(pts[m, 0], pts[m, 1], c=c, s=90, label=doc_name,
                   edgecolors="black", linewidths=0.5)
    ax.set_title(f"{name} space (PCA)")
axes[1].legend(fontsize=8, loc="best")
plt.tight_layout(); plt.show()

---
## Section 4 — FAISS: the vector index

Notebook 1's search was a Python for-loop over every chunk — **exact brute force, O(n) per query**.
Correct, but at 1M vectors it's far too slow.

FAISS is what a vector database uses underneath. Two index types worth knowing:

| Index | Search | Recall | Use when |
|---|---|---|---|
| `IndexFlatIP` | exact, O(n) | 100% | < ~100k vectors |
| `IndexHNSWFlat` | approximate graph walk | ~95–99% | millions of vectors |

**Critical trade-off to internalize:** approximate indexes (HNSW, IVF) trade *recall* for *speed*.
They can silently miss the true nearest neighbor. With 10 chunks we use `IndexFlatIP` — exact.
`pgvector` in Milestone 13 of the main project uses HNSW, and that recall loss becomes a real
tuning parameter you must measure.


In [ ]:
dim = X_dense.shape[1]                # 384 — the vector dimension every chunk was pooled down to
index = faiss.IndexFlatIP(dim)         # IP = Inner Product. FAISS needs to know the vector size
                                        # up front so it can allocate storage; it doesn't hold ANY
                                        # vectors yet — .add() below is what actually populates it.
                                        # "Flat" means exact search: no approximation, no graph —
                                        # it just compares the query against every stored vector,
                                        # same algorithm as Notebook 1's for-loop, only implemented
                                        # in fast compiled code instead of a Python loop.
index.add(X_dense)                     # load all chunk vectors into the index, in row order —
                                        # FAISS remembers them by POSITION (0, 1, 2, ...), which is
                                        # why the ids returned by search() are just row numbers,
                                        # not chunk_ids — we map row number back to chunk_id ourselves.
print(f"FAISS index: {index.ntotal} vectors × {dim} dims")

def faiss_search(question: str, k: int = 3, candidate_ids=None):
    qv = embedder.encode([question], normalize_embeddings=True).astype("float32")
    if candidate_ids is None:
        # index.search() returns TWO arrays: scores (similarity to each result) and ids (their
        # row positions in the index). Both come back as (1, k) because we passed a batch of
        # ONE query — [0] unwraps that outer batch dimension down to plain (k,) arrays.
        scores, ids = index.search(qv, k)
        ids, scores = ids[0], scores[0]
    else:
        # Metadata pre-filtering (Section 7) needs to search only a SUBSET of rows, which FAISS's
        # index API doesn't support directly — so for the filtered case we fall back to the same
        # brute-force numpy approach as dense_scores(), just restricted to candidate_ids rows.
        sub = X_dense[candidate_ids]                    # pull out only the allowed rows
        s = sub @ qv.ravel()                             # dot product against just that subset
        order = np.argsort(s)[::-1][:k]                  # top-k positions WITHIN the subset
        ids, scores = np.array(candidate_ids)[order], s[order]   # map back to ORIGINAL row numbers
    return pd.DataFrame({
        "score": scores, "chunk_id": df.loc[ids, "chunk_id"].values,   # .loc[ids] looks up the
        "doc": df.loc[ids, "doc"].values, "text": df.loc[ids, "text"].values,   # original rows by
        "row": ids,                                                              # the row numbers FAISS returned
    })

# Verify FAISS agrees with our hand-written numpy search
q = "How do I restart the payment service?"
faiss_top = faiss_search(q, k=3)
manual_top = np.argsort(dense_scores(q))[::-1][:3]
print(f"\nFAISS rows : {faiss_top.row.tolist()}")
print(f"Manual rows: {manual_top.tolist()}")
print("✅ identical — FAISS is an accelerator, not a different algorithm"
      if faiss_top.row.tolist() == manual_top.tolist() else "⚠️ mismatch")
faiss_top[["score", "chunk_id", "doc"]]

---
## Section 5 — Cross-Encoder Re-ranking

Even with good embeddings, one failure survives: **vocabulary collision**. In Notebook 1,
*"Customers see failures at checkout"* retrieved the Kubernetes runbook because "failures" is strong there.

**Why this happens — bi-encoder vs cross-encoder:**

```
BI-ENCODER (what we've used so far)
  question ──► [encoder] ──► vector ─┐
                                     ├─► cosine
  chunk    ──► [encoder] ──► vector ─┘
  The two texts NEVER see each other. Chunk vectors are precomputed → fast, scales to millions.

CROSS-ENCODER (re-ranking)
  [question + chunk together] ──► [encoder] ──► relevance score
  Full attention BETWEEN question and chunk → far more accurate,
  but requires one forward pass PER PAIR → cannot pre-compute, doesn't scale.
```

**So we use both — the standard two-stage retrieval pattern:**
1. **Retrieve** top-20 cheaply with the bi-encoder + FAISS (fast, high recall, mediocre precision)
2. **Re-rank** those 20 with the cross-encoder (slow, high precision, tiny candidate set)

This is exactly how production search works — Google, Bing, and every serious RAG system.


In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
# Unlike embedder.encode(), this model does NOT produce a reusable vector per text. Its .predict()
# takes PAIRS of (question, chunk) and runs them THROUGH THE MODEL TOGETHER, one forward pass per
# pair, outputting a single relevance score. That's the "full attention between question and chunk"
# from the diagram above — it's why cross-encoders can't be pre-computed like the FAISS index can.

def search_rerank(question: str, k: int = 3, retrieve_k: int = 20):
    """Stage 1: bi-encoder retrieval. Stage 2: cross-encoder re-ranking."""
    stage1 = faiss_search(question, k=min(retrieve_k, len(df)))    # cheap: get a WIDE candidate pool
    pairs = [[question, t] for t in stage1["text"]]                # build (question, chunk_text) pairs —
                                                                     # one pair per Stage-1 candidate, NOT
                                                                     # one call per chunk in the whole corpus
    stage1 = stage1.assign(rerank_score=reranker.predict(pairs))   # one forward pass per pair -> one score
    stage1["stage1_rank"] = range(1, len(stage1) + 1)              # remember the ORIGINAL FAISS ranking
    out = stage1.sort_values("rerank_score", ascending=False).head(k).reset_index(drop=True)
    out["stage2_rank"] = range(1, len(out) + 1)                    # and the NEW rank after re-ranking,
    return out                                                     # so we can compare the two below

collision_q = "Customers see failures at checkout"
before = faiss_search(collision_q, k=5)      # Stage 1 only — the bi-encoder's opinion
after  = search_rerank(collision_q, k=5)     # Stage 1 + Stage 2 — the cross-encoder's correction

print(f'QUERY: "{collision_q}"   (expected: payment-service)\n')
print("STAGE 1 — bi-encoder (FAISS):")
for _, r in before.iterrows():
    print(f"   {r.score:+.3f}  {r.doc}")
print("\nSTAGE 2 — after cross-encoder re-ranking:")
for _, r in after.iterrows():
    print(f"   {r.rerank_score:+.3f}  {r.doc}   (was rank {r.stage1_rank})")

In [ ]:
# Visualize the rank shuffle: for each chunk that survived into the top-k, draw a line from its
# Stage-1 (bi-encoder) rank to its Stage-2 (cross-encoder) rank — a line that moves UP shows the
# cross-encoder promoting a chunk the bi-encoder had ranked lower.
fig, ax = plt.subplots(figsize=(9, 5))
for _, r in after.iterrows():
    color = "#55A868" if r.doc == "payment-service" else "#BBBBBB"
    ax.plot([0, 1], [r.stage1_rank, r.stage2_rank], "o-", color=color, lw=2, ms=9)
    ax.annotate(r.doc, (1.02, r.stage2_rank), va="center", fontsize=9)
ax.set_xticks([0, 1]); ax.set_xticklabels(["bi-encoder rank", "cross-encoder rank"])
ax.invert_yaxis(); ax.set_ylabel("rank"); ax.set_xlim(-0.1, 1.6)
ax.set_title("Re-ranking reshuffles the candidate set (green = correct runbook)")
plt.tight_layout(); plt.show()

In [ ]:
def rerank_scores(q):
    """Adapter so evaluate() can score the reranked pipeline over all chunks."""
    res = search_rerank(q, k=len(df), retrieve_k=len(df))
    s = np.zeros(len(df))
    for _, r in res.iterrows():
        s[int(r.row)] = r.rerank_score
    return s

res_rerank = evaluate(rerank_scores, "MiniLM + cross-encoder re-ranking")

**Cost check — always quantify the latency you just bought.** The cell below measures it.
Re-ranking 20 candidates means 20 extra transformer forward passes on the critical path of every query.


In [ ]:
import time

def timeit(fn, q, n=3):
    fn(q)                                   # warm up
    t0 = time.perf_counter()
    for _ in range(n): fn(q)
    return (time.perf_counter() - t0) / n * 1000

q = "Why is Kafka consumer lag increasing?"
t_bi = timeit(lambda x: faiss_search(x, k=3), q)
t_cross = timeit(lambda x: search_rerank(x, k=3, retrieve_k=20), q)
print(f"bi-encoder + FAISS      : {t_bi:7.1f} ms")
print(f"+ cross-encoder rerank  : {t_cross:7.1f} ms   ({t_cross/t_bi:.1f}× slower)")
print("\nIs the accuracy worth the latency? That is a product decision, and")
print("during an incident an SRE will happily wait 300ms for a better answer.")

---
## Section 6 — Hybrid Search (BM25 + Vector)

Dense embeddings have their *own* blind spot, and it is the mirror image of TF-IDF's.

**Semantic models are bad at exact identifiers.** Query `payment-api-0` or `NotEnoughReplicasException`
and MiniLM gives you something *semantically similar* — but an on-call engineer pasting an exact
error string wants **that literal token**. Neural embeddings blur precise strings; keyword search nails them.

This matters enormously for SRE work, where queries are full of exact pod names, error classes,
and topic names.

**BM25** is the modern refinement of TF-IDF (term saturation + document-length normalization).
Hybrid search runs both and fuses the rankings.

### Fusing scores: why not just add them?
BM25 scores are unbounded (0 → 20+); cosine scores are bounded (−1 → 1). Adding them directly lets
BM25 dominate arbitrarily. Two standard fixes:
- **Normalize then weight**: `α · norm(dense) + (1−α) · norm(bm25)` — simple, but sensitive to score outliers.
- **Reciprocal Rank Fusion (RRF)**: ignore scores, use *ranks*: `Σ 1/(k + rank)`. Robust, no tuning. We show both.


In [ ]:
def tokenize(t):
    # BM25's library wants a plain list of lowercase word-tokens per document — much simpler
    # than TfidfVectorizer's regex-based tokenizer, but the same basic idea: split text into words.
    return re.findall(r"[a-z0-9][a-z0-9\-_.]+", t.lower())

# BM25Okapi is built from the WHOLE corpus's tokenized chunks up front — like TfidfVectorizer's
# fit(), it needs to see every document to compute term frequencies and inverse document frequencies.
bm25 = BM25Okapi([tokenize(t) for t in df["text"]])

def bm25_scores(q):
    return np.asarray(bm25.get_scores(tokenize(q)))   # scores for the query against EVERY chunk at once

# The exact-identifier query that dense retrieval fumbles
exact_q = "NotEnoughReplicasException"
b, d = bm25_scores(exact_q), dense_scores(exact_q)
print(f'QUERY: "{exact_q}"  (a literal exception string; expected: kafka-cluster)\n')
print(f"BM25   best → {df.loc[int(b.argmax()), 'doc']:20s} score {b.max():.3f}")
print(f"MiniLM best → {df.loc[int(d.argmax()), 'doc']:20s} score {d.max():.3f}")
print("\nPer-chunk scores:")
pd.DataFrame({"chunk_id": df.chunk_id, "doc": df.doc,
              "bm25": b.round(2), "minilm": d.round(3)}).sort_values("bm25", ascending=False).head(5)

In [ ]:
def minmax(x):
    """Rescale any array to the 0-1 range, so BM25's unbounded scores (0 to 20+) and MiniLM's
    bounded cosine scores (-1 to 1) become comparable before we average them together."""
    rng = x.max() - x.min()
    return np.zeros_like(x) if rng < 1e-9 else (x - x.min()) / rng   # guard: if every score is
                                                                       # identical, rng=0 and we'd
                                                                       # divide by zero — return all-0s instead

def hybrid_scores(q, alpha=0.5):
    """alpha=1.0 → pure dense; alpha=0.0 → pure BM25."""
    return alpha * minmax(dense_scores(q)) + (1 - alpha) * minmax(bm25_scores(q))

def rrf_scores(q, k=60):
    """Reciprocal Rank Fusion — fuse by RANK POSITION, not by raw score, so the two retrievers'
    very different score scales never need normalizing at all."""
    out = np.zeros(len(df))
    for s in (dense_scores(q), bm25_scores(q)):
        # argsort() once gives the INDICES that would sort s ascending. argsort() a SECOND time
        # gives, for each original position, its RANK in that sorted order — this is the standard
        # "double argsort" trick for turning scores into 0-indexed ranks without a manual sort loop.
        # We negate s first so the HIGHEST score gets rank 0 (best), not the lowest.
        ranks = (-s).argsort().argsort()          # 0 = best, 1 = second best, ...
        out += 1.0 / (k + ranks + 1)              # RRF formula: 1/(k + rank + 1). The constant k=60
                                                    # (a standard default) softens the difference between
                                                    # e.g. rank 0 and rank 1 so one retriever's #1 pick
                                                    # doesn't automatically crush the other's #1 pick.
    return out

res_bm25   = evaluate(bm25_scores, "BM25 only")
print()
res_hybrid = evaluate(lambda q: hybrid_scores(q, 0.5), "Hybrid (alpha=0.5, min-max)")
print()
res_rrf    = evaluate(rrf_scores, "Hybrid (Reciprocal Rank Fusion)")

### 6.1 Sweep alpha — find the blend, and see each half of the eval set pull in opposite directions


In [ ]:
alphas = np.linspace(0, 1, 11)
rows = []
for a in alphas:
    hits = {"lexical": [], "semantic": []}
    for q, expected, kind in TEST_SET:
        best = int(np.argmax(hybrid_scores(q, a)))
        hits[kind].append(df.loc[best, "doc"] == expected)
    rows.append({"alpha": a,
                 "lexical": np.mean(hits["lexical"]),
                 "semantic": np.mean(hits["semantic"]),
                 "overall": np.mean(hits["lexical"] + hits["semantic"])})
df_alpha = pd.DataFrame(rows)

plt.plot(df_alpha.alpha, df_alpha.lexical,  "o-", label="lexical questions")
plt.plot(df_alpha.alpha, df_alpha.semantic, "s-", label="semantic questions")
plt.plot(df_alpha.alpha, df_alpha.overall,  "k--", lw=2, label="overall")
plt.xlabel("alpha  (0 = pure BM25   →   1 = pure dense)")
plt.ylabel("hit@1"); plt.ylim(-0.05, 1.05); plt.grid(alpha=0.3); plt.legend()
plt.title("Hybrid search: the blend is a tunable trade-off")
plt.tight_layout(); plt.show()
print(df_alpha.to_string(index=False))

**How to read this chart.** If the two curves slope in opposite directions, you are looking at the
central trade-off of hybrid search: keyword matching wins on literal queries, embeddings win on
paraphrases, and alpha buys one with the other. The best alpha is a property of **your query mix**,
not a universal constant — which is why you must build an eval set from *real* on-call questions
before tuning it. Do not copy someone's alpha from a blog post.


---
## Section 7 — Metadata Filtering

Everything so far searches **all** chunks. But during a Kafka incident, searching login runbooks is
pure noise — and worse, it's a chance to retrieve something plausible-but-wrong.

If the SRE Copilot's Incident Bundle already says `"service": "payment-service"`, we should
**pre-filter** to that service's runbooks and search only within them.

**Pre-filter vs post-filter — a real production distinction:**
- **Post-filter**: retrieve top-k globally, then drop non-matching results. *Bug:* if all top-k belong
  to other services, you get back **nothing**.
- **Pre-filter**: restrict the candidate set *first*, then search within it. Always returns k results.

Real vector DBs implement pre-filtering natively (pgvector: a `WHERE` clause alongside the ANN scan).
Here we do it by slicing the candidate list, which is exactly what `faiss_search(candidate_ids=...)` supports.


In [ ]:
def filtered_search(question, k=3, service=None, exclude=None):
    mask = np.ones(len(df), dtype=bool)      # start with "every chunk is allowed" (all True)
    if service:
        # df.doc.isin([...]) returns a boolean Series: True where the chunk's doc is in the allowed
        # list. isinstance check lets `service` be either one string or a list of strings.
        # `&=` combines with the existing mask using AND — narrowing, never widening, the candidate set.
        mask &= df.doc.isin([service] if isinstance(service, str) else service).values
    if exclude:
        mask &= ~df.doc.isin([exclude] if isinstance(exclude, str) else exclude).values  # ~ = boolean NOT
    ids = np.where(mask)[0]     # convert the True/False mask into actual ROW NUMBERS where it's True
    if len(ids) == 0:           # filter was too strict — nothing survived, return an empty result
        return pd.DataFrame(columns=["score", "chunk_id", "doc", "text", "row"])
    return faiss_search(question, k=min(k, len(ids)), candidate_ids=ids)   # search only within ids

q = "How do I restart it?"       # deliberately vague — exactly how humans talk during an incident
print(f'QUERY: "{q}"\n')
print("NO FILTER (ambiguous — every runbook has a restart procedure):")
for _, r in faiss_search(q, k=3).iterrows():
    print(f"   {r.score:+.3f}  {r.doc}")

for svc in ["payment-service", "kafka-cluster"]:
    print(f"\nPRE-FILTERED to {svc} (Incident Bundle told us the service):")
    for _, r in filtered_search(q, k=3, service=svc).iterrows():
        print(f"   {r.score:+.3f}  {r.chunk_id}")

**Connect this to the main project.** In the SRE Copilot, Flink emits an Incident Bundle
containing `service`, `severity`, and `deployment`. That bundle is not just LLM input — it is a
**retrieval filter**. Milestone 16 (Retrieval Tools) should pass those fields down as metadata
filters, which cuts both cost and hallucination risk. Filtering is free accuracy; use it whenever
structured context is available.


---
## Section 8 — The Full Pipeline, Assembled

Every component, in production order:

```
question
   │
   ├── metadata filter        ← narrow candidates using Incident Bundle fields
   │
   ├── STAGE 1: retrieve      ← bi-encoder + FAISS (top-20) ⊕ BM25 (top-20), fused by RRF
   │
   ├── STAGE 2: re-rank       ← cross-encoder over the ~20 candidates → top-3
   │
   ├── build prompt           ← grounded, cited, with a refusal escape hatch
   │
   └── LLM answer
```


In [ ]:
def rag_pipeline(question, k=3, service=None, retrieve_k=20, alpha=0.5, use_rerank=True):
    # ---- metadata pre-filter (same pattern as filtered_search above) --------------
    mask = np.ones(len(df), dtype=bool)
    if service:
        mask &= df.doc.isin([service] if isinstance(service, str) else service).values
    ids = np.where(mask)[0]           # row numbers of chunks allowed to compete for this query
    if len(ids) == 0:
        return pd.DataFrame(columns=["chunk_id", "doc", "text", "final_score"])

    # ---- stage 1: hybrid retrieval, but ONLY over the filtered subset -------------
    d = dense_scores(question)[ids]   # score every chunk, then slice down to just the allowed ones
    b = bm25_scores(question)[ids]    # (simpler than re-deriving a filtered index — fine at this scale)
    fused = np.zeros(len(ids))
    for s in (d, b):                                  # same RRF fusion as rrf_scores() above, just
        ranks = (-s).argsort().argsort()               # inlined here so this function is self-contained
        fused += 1.0 / (60 + ranks + 1)
    # take the top retrieve_k candidates by fused score, but ids[...] maps back from "position within
    # the filtered subset" to the ORIGINAL row number in df — this indirection is easy to get backwards
    cand = ids[np.argsort(fused)[::-1][:min(retrieve_k, len(ids))]]
    stage1 = df.loc[cand].copy()
    stage1["stage1_score"] = np.sort(fused)[::-1][:len(cand)]

    # ---- stage 2: cross-encoder re-rank the (small) candidate set -----------------
    if use_rerank and len(cand) > 1:
        stage1["final_score"] = reranker.predict([[question, t] for t in stage1.text])
    else:
        stage1["final_score"] = stage1["stage1_score"]     # skip reranking if disabled or only 1 candidate
    return stage1.sort_values("final_score", ascending=False).head(k).reset_index(drop=True)


def build_prompt(question, k=3, service=None):
    hits = rag_pipeline(question, k=k, service=service)     # run the full pipeline above
    if hits.empty:
        return None, hits                                    # nothing retrieved — caller must handle this
    ctx = "\n\n".join(f"[Source: {r.chunk_id} | relevance={r.final_score:.2f}]\n{r.text}"
                       for _, r in hits.iterrows())          # same prompt-assembly pattern as Notebook 1
    prompt = f"""You are an SRE assistant. Answer the on-call engineer's question using ONLY the context below.
Cite the source chunk id for every instruction you give.
If the context does not contain the answer, say "I don't have a runbook for that" — do not guess.

=== CONTEXT ===
{ctx}

=== QUESTION ===
{question}

=== ANSWER ==="""
    return prompt, hits

prompt, hits = build_prompt("The message bus is falling behind, what do I check?", k=3)
print(prompt)
print(f"\n--- {len(prompt)} chars ≈ {len(prompt)//4} tokens ---")

In [ ]:
def mock_llm(prompt: str) -> str:
    """Same free mock as Notebook 1 — proves the pipeline end-to-end with no API key."""
    context = prompt.split("=== CONTEXT ===")[1].split("=== QUESTION ===")[0]   # slice out just the context block
    pattern = r"(?:kubectl|redis-cli|systemctl|kafka-[\w.\-]+)(?:\s+(?![A-Z])[\w./<>#=\-]+)*"
    cmds = list(dict.fromkeys(re.findall(pattern, context)))    # find + de-dupe (order-preserving) commands
    if not cmds:
        return "I don't have a runbook for that."
    return "Based on the retrieved runbooks:\n" + "\n".join(f"{i+1}. `{c}`" for i, c in enumerate(cmds))

# Three test questions, each exercising a different part of the pipeline:
#   1. paraphrase (no service filter)   2. vague question WITH a service filter   3. genuinely out of scope
for q, svc in [("The message bus is falling behind, what do I check?", None),
               ("How do I restart it?", "payment-service"),
               ("How do I rotate the TLS certificate on the mail server?", None)]:
    p, _ = build_prompt(q, k=3, service=svc)
    print(f"Q: {q}   (filter={svc})")
    print(textwrap.indent(mock_llm(p), "   "), "\n")

### 8.1 The out-of-scope query — a hallucination path we just left open

Look hard at that **third** query: *"How do I rotate the TLS certificate on the mail server?"*
We have **no mail-server runbook**. The correct answer is "I don't have a runbook for that."

But retrieval returned three chunks anyway, and the LLM dutifully produced commands from them.
This is *not* an LLM failure — the model did exactly what we asked, using the context we handed it.
**The retrieval layer lied first.**

The root cause is structural: `top-k` **always returns k results**. Ranking is *relative*; it will
happily hand you the least-bad chunk out of a corpus of entirely irrelevant chunks. Nothing in a
top-k search ever asks *"is this actually relevant?"*

**The fix: an absolute relevance threshold.** Retrieve top-k, then discard anything below a score
floor. If nothing survives, return no context at all and let the model refuse.

Note the trade-off this introduces — it is the same precision/recall dial as every alerting system
you've ever tuned as an SRE:
- threshold **too high** → the system refuses questions it *could* have answered (false negatives)
- threshold **too low** → hallucination path stays open (false positives)


In [ ]:
def rag_pipeline_thresholded(question, k=3, service=None, min_score=None):
    hits = rag_pipeline(question, k=k, service=service)     # get the normal top-k results first
    if min_score is not None and not hits.empty:
        hits = hits[hits.final_score >= min_score]           # THEN drop anything below the floor —
    return hits                                               # this can shrink hits to fewer than k,
                                                               # or even to ZERO rows, which is the point

def answer(question, k=3, service=None, min_score=None):
    hits = rag_pipeline_thresholded(question, k, service, min_score)
    if hits.empty:
        return "I don't have a runbook for that.", hits    # refuse BEFORE ever building a prompt —
                                                              # the LLM never even sees this question
    ctx = "\n\n".join(f"[Source: {r.chunk_id} | relevance={r.final_score:.2f}]\n{r.text}"
                       for _, r in hits.iterrows())
    prompt = f"""You are an SRE assistant. Answer using ONLY the context below.
If the context does not contain the answer, say "I don't have a runbook for that".

=== CONTEXT ===
{ctx}

=== QUESTION ===
{question}

=== ANSWER ==="""
    return mock_llm(prompt), hits

# First: what do in-scope vs out-of-scope questions actually score?
print("Top relevance score per question — find the separation:\n")
probe_set = [("How do I restart the payment service?", "in-scope"),
             ("Why is Kafka consumer lag increasing?", "in-scope"),
             ("The message bus is falling behind",     "in-scope"),
             ("How do I rotate the TLS certificate on the mail server?", "OUT-of-scope"),
             ("What is the refund policy for enterprise customers?",      "OUT-of-scope"),
             ("How do I bake sourdough bread?",                           "OUT-of-scope")]
for q, tag in probe_set:
    h = rag_pipeline(q, k=1)               # k=1: we only care about the SINGLE best score per question
    top = float(h.final_score.iloc[0]) if not h.empty else float("nan")
    print(f"  {top:+7.3f}  [{tag:12s}]  {q}")

print("\n" + "="*70)
print("Pick a threshold BETWEEN the two groups, then re-run:\n")
THRESHOLD = 0.0      # ← TUNE ME using the scores printed above
for q, tag in probe_set:
    ans, _ = answer(q, k=3, min_score=THRESHOLD)
    refused = ans.startswith("I don't have")                       # did the pipeline refuse to answer?
    verdict = "refused " if refused else "answered"
    # correct behaviour: refuse IFF the question is actually out-of-scope — this XNOR-style check
    # is True when both sides agree (refused+OUT-of-scope, or answered+in-scope)
    correct = "✅" if (refused == (tag == "OUT-of-scope")) else "❌"
    print(f"  {correct} {verdict}  [{tag:12s}]  {q}")

**Your task — this is the most valuable experiment in the notebook.** Read the printed scores,
pick a `THRESHOLD` that separates in-scope from out-of-scope, re-run, and try to get all six ✅.

Then find the uncomfortable part: try to break your own threshold. Add an out-of-scope question that
happens to share vocabulary with a runbook (*"How do I restart my laptop?"* — note "restart"). Does
your threshold still hold? If a single well-chosen question defeats it, you've learned why
**production RAG systems need calibrated confidence, not a hand-picked constant** — and why the
SRE Copilot's Milestone 20 (Confidence Score) is its own dedicated milestone rather than one line of code.

**Production practice:** log the top relevance score for every query. When that distribution drifts,
your corpus and your traffic have diverged — which is a *retrieval* alarm, and it fires long before
users start complaining about wrong answers.


---
## Section 9 — Final Scoreboard

Every configuration, measured on the same eval set. This table is the deliverable of the notebook.


In [ ]:
# Every retriever built in this notebook, keyed by display name -> its score_fn(question) function.
# Note rerank_scores isn't defined inline here — it was built back in Section 5 as an "adapter"
# that runs the full rerank pipeline but returns a score for EVERY chunk (not just the top-k),
# so it fits the same score_fn(question) -> array-of-scores shape as everything else.
configs = {
    "TF-IDF (Notebook 1)":       tfidf_scores,
    "BM25":                      bm25_scores,
    "MiniLM dense":              dense_scores,
    "Hybrid (alpha=0.5)":        lambda q: hybrid_scores(q, 0.5),
    "Hybrid (RRF)":              rrf_scores,
    "MiniLM + cross-encoder":    rerank_scores,
}

rows = []
for name, fn in configs.items():
    hits = {"lexical": [], "semantic": []}
    for q, expected, kind in TEST_SET:
        best = int(np.argmax(fn(q)))                       # top-scoring chunk for this retriever
        hits[kind].append(df.loc[best, "doc"] == expected)  # True/False, bucketed by question kind
    rows.append({"config": name,
                 "lexical": np.mean(hits["lexical"]),        # fraction correct on lexical questions
                 "semantic": np.mean(hits["semantic"]),      # fraction correct on semantic questions
                 "overall": np.mean(hits["lexical"] + hits["semantic"])})   # list concat, not sum —
                                                                              # combines both lists before averaging
board = pd.DataFrame(rows)
print(board.to_string(index=False))

x = np.arange(len(board)); w = 0.27         # x = bar group positions, w = width of each individual bar
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w, board.lexical,  w, label="lexical",  color="#4C72B0")   # three bars per config, offset
ax.bar(x,     board.semantic, w, label="semantic", color="#DD8452")   # left/center/right by -w/0/+w
ax.bar(x + w, board.overall,  w, label="overall",  color="#55A868")
ax.set_xticks(x); ax.set_xticklabels(board.config, rotation=20, ha="right")
ax.set_ylabel("hit@1"); ax.set_ylim(0, 1.05); ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title("Retrieval configurations on the same evaluation set")
plt.tight_layout(); plt.show()

> ⚠️ **A word of caution about this scoreboard.** Ten questions over ten chunks is a *demonstration*,
> not an experiment. One question flipping moves hit@1 by 0.10, which is well inside the noise.
> Do not conclude "RRF beats alpha=0.5 in general" from this table. What it legitimately teaches is
> the **method**: split your eval set by failure mode, change one component at a time, and read the
> per-category numbers rather than the aggregate. Notebook 3 replaces this with proper evaluation
> (RAGAS, larger sets, confidence intervals).


---
## Conclusion & Quiz

**What changed from Notebook 1:**

| | Notebook 1 | Notebook 2 |
|---|---|---|
| Embedding | TF-IDF (readable, lexical) | MiniLM (learned, semantic) |
| Index | Python list + for-loop | FAISS `IndexFlatIP` |
| Ranking | cosine top-k | + cross-encoder re-rank |
| Matching | vocabulary only | + BM25 hybrid fusion |
| Scope | whole corpus | + metadata pre-filtering |

**The transferable lesson** is not "use MiniLM". It's the diagnostic loop:
*build an eval set that separates failure modes → change one component → read the per-category
numbers → fix the component that's actually the bottleneck.* That loop is what separates engineers
who tune RAG systems from engineers who guess at them.

### Quiz — answer before Notebook 3

1. We passed `normalize_embeddings=True` and then used FAISS `IndexFlatIP` (inner product). Explain
   precisely why those two choices must go together, and what would silently break if you dropped
   the normalization but kept `IndexFlatIP`.
2. A cross-encoder is more accurate than a bi-encoder. Why can't we just cross-encode the entire
   corpus and skip the bi-encoder stage? Give the complexity argument.
3. Your production query logs show most on-call engineers paste **exact error strings** into the
   search box. Which direction should you move `alpha`, and why?
4. Explain the difference between pre-filtering and post-filtering by metadata, and describe a
   concrete scenario where post-filtering returns **zero** results while pre-filtering returns three
   good ones.
5. You upgrade your embedding model from MiniLM to a larger one but forget to re-index the corpus.
   Retrieval throws no error. What exactly is being compared, and why are the results garbage?
   *(This is the same failure as Notebook 1's quiz #2 — make sure you can now explain it in terms of vector spaces.)*
6. In §8.1 the pipeline answered a question about a mail server we have no runbook for. Explain why
   *top-k retrieval can never refuse on its own*, and why a relevance threshold is a property of the
   **retrieval** layer rather than something you can prompt the LLM out of.
7. **Design question, no single right answer:** the SRE Copilot's Incident Bundle contains `service`,
   `severity`, `deployment`, and `topExceptions`. Which of these would you use as metadata *filters*,
   which as part of the *query text*, and which as *neither*? Justify each.

**Next — Notebook 3: Production RAG.** PostgreSQL + pgvector, HNSW indexing and its recall trade-off,
proper evaluation with RAGAS, and observability. Then Notebook 4 wires retrieval into LangGraph as an
agent tool, which is where this work merges back into Milestones 13 and 16 of the main project.
